In [ ]:
"""
GPU Benchmarking on Google Colab

This notebook runs the GPU benchmark on Colab's free NVIDIA GPU.
No setup required - just open and click "Run all".

Upload Instructions:
1. Go to: https://colab.research.google.com
2. Click "Upload notebook" and select this file
3. Or paste the code from this notebook directly into Colab cells
"""

# ============================================================================
# CELL 1: Install and Import Dependencies
# ============================================================================

# Install required packages
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tensorflow[and-cuda]==2.13.1"])

import os
import json
import time
import numpy as np
import tensorflow as tf
from datetime import datetime

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices('GPU'))
print("CUDA available:", tf.test.is_built_with_cuda())

# ============================================================================
# CELL 2: Verify GPU is Available
# ============================================================================

gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError("No GPU found! Make sure Colab GPU is enabled: Runtime > Change runtime type > GPU")

print(f"✓ Found {len(gpus)} GPU(s)")
for gpu in gpus:
    print(f"  - {gpu}")

# ============================================================================
# CELL 3: Upload Model Files
# ============================================================================

# Mount Google Drive to upload files
from google.colab import drive
from google.colab import files

drive.mount('/content/gdrive')

# Or directly upload from your machine
print("Upload your benchmark files (DNN/, RNN/, CNN/, data/ folders)")
print("and benchmark_native.py")
uploaded = files.upload()

# ============================================================================
# CELL 4: Extract and Setup Files
# ============================================================================

import zipfile
import shutil

# If you uploaded a zip file, extract it
for filename in uploaded:
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall()

# Create results directory
os.makedirs('results', exist_ok=True)

# ============================================================================
# CELL 5: Run Benchmark (All 15 Models, 1 Trial)
# ============================================================================

import subprocess
import os

os.chdir('/content')

# Run benchmark for all models with 1 trial on GPU
result = subprocess.run([
    'python', 'benchmark_native.py',
    '--trials', '1',
    '--output', 'results/benchmark_results_gpu.csv'
], capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

# ============================================================================
# CELL 6: View Results
# ============================================================================

import pandas as pd

# Read and display results
csv_file = 'results/benchmark_results_gpu.csv'
if os.path.exists(csv_file):
    df = pd.read_csv(csv_file)
    print("\n✓ Benchmark Results (GPU):\n")
    print(df.to_string())
    print(f"\nCSV saved to: {csv_file}")
    print(f"Total rows: {len(df)}")
else:
    print("Results file not found")

# ============================================================================
# CELL 7: Download Results
# ============================================================================

from google.colab import files

# Download results to your machine
print("Downloading results...")
files.download('results/benchmark_results_gpu.csv')
print("✓ Download complete")

"""
QUICK START:
1. Go to https://colab.research.google.com
2. Click "New notebook" 
3. Copy this entire script into the first cell and run it
4. When prompted to upload, select:
   - Your benchmark_native.py file
   - DNN/ folder (as zip or individual files)
   - RNN/ folder (as zip or individual files)  
   - CNN/ folder (as zip or individual files)
   - data/ folder (with data_cifar10.py)
5. Results will be available for download at the end

Expected Runtime: ~30-40 minutes for all 15 models
GPU: NVIDIA Tesla K80 or similar (Colab's free tier)
"""